# Online News Popularity — analysis companion

Read the website for the full reasoning. The canonical executable is `src/analyze.py`. Outputs below are from an executed run of the inspection and metric-verification cells against the saved canonical results. Full model training is performed by the analysis script, not repeated inside this notebook.

Execution method: Python 3.13.9 with IPython 9.7.0 in a shared in-process namespace. See VERIFICATION.md for the separate pipeline reproduction check.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
s = json.loads((ROOT / 'results/summary.json').read_text())
s['environment'], s['csv_sha256']

({'python': '3.13.9',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'scikit-learn': '1.7.2'},
 'b66d9088632308cc27fa35af847650d174a5a50503987c4e511de94a99d1c218')

## Split and provenance
Date groups, rather than rows, define the 60/20/20 partitions. The test is never used for tuning. The original row ID allows exact tracing to the supplied CSV.

In [2]:
splits = pd.read_csv(ROOT / 'results/splits.csv')
assert splits.groupby('date').split.nunique().max() == 1
s['splits']

{'train': {'n': 21892, 'start': '2013-01-07', 'end': '2014-03-14'},
 'validation': {'n': 8440, 'start': '2014-03-15', 'end': '2014-08-05'},
 'test': {'n': 9312, 'start': '2014-08-06', 'end': '2014-12-27'}}

In [3]:
s['invalid_ratios'], s['zero_content'], s['features'], s['excluded']

({'n_unique_tokens': 1, 'n_non_stop_words': 1, 'n_non_stop_unique_tokens': 1},
 1181,
 ['n_tokens_title',
  'n_tokens_content',
  'n_unique_tokens',
  'n_non_stop_words',
  'n_non_stop_unique_tokens',
  'num_hrefs',
  'num_self_hrefs',
  'num_imgs',
  'num_videos',
  'average_token_length',
  'num_keywords',
  'data_channel_is_lifestyle',
  'data_channel_is_entertainment',
  'data_channel_is_bus',
  'data_channel_is_socmed',
  'data_channel_is_tech',
  'data_channel_is_world',
  'weekday_is_monday',
  'weekday_is_tuesday',
  'weekday_is_wednesday',
  'weekday_is_thursday',
  'weekday_is_friday',
  'weekday_is_saturday',
  'weekday_is_sunday',
  'is_weekend',
  'global_subjectivity',
  'global_sentiment_polarity',
  'global_rate_positive_words',
  'global_rate_negative_words',
  'rate_positive_words',
  'rate_negative_words',
  'avg_positive_polarity',
  'min_positive_polarity',
  'max_positive_polarity',
  'avg_negative_polarity',
  'min_negative_polarity',
  'max_negative_polarity',
 

## All candidate comparisons
The selected forest leaf size is 30. After choices are frozen, all final families are refitted on train+validation. This is why final test scores are not from the training-only validation fits.

In [4]:
pd.read_json(ROOT / 'results/validation_trials.json')

,model,parameters,validation_log_mae
0,Median baseline,{},0.643287
1,Ridge,{'alpha': 1},0.605963
2,Ridge,{'alpha': 100},0.606091
3,Random forest,{'min_samples_leaf': 10},0.607717
4,Random forest,{'min_samples_leaf': 30},0.603777
5,Boosted trees,{'max_leaf_nodes': 15},0.606234
6,Boosted trees,{'max_leaf_nodes': 31},0.606035


In [5]:
pd.read_csv(ROOT / 'results/metrics.csv')

,model,validation_log_mae,test_log_mae,test_rmsle,test_mae_shares,test_r2_log,parameters
0,Median baseline,0.643287,0.656731,0.910180,2153.250430,-0.008669,{}
1,Ridge,0.605963,0.638682,0.862901,2130.694492,0.093399,{'alpha': 1}
2,Random forest,0.603777,0.631713,0.854091,2121.328535,0.111817,{'min_samples_leaf': 30}
3,Boosted trees,0.606035,0.633445,0.854676,2123.997869,0.110600,{'max_leaf_nodes': 31}


## Independently recompute the primary metric

In [6]:
pred = pd.read_csv(ROOT / 'results/test_predictions.csv')
for result in s['metrics']:
    actual = np.abs(np.log1p(pred.shares) - np.log1p(pred[result['model']])).mean()
    assert np.isclose(actual, result['test_log_mae'])
    print(result['model'], actual)

Median baseline 0.6567312341203634
Ridge 0.6386818212765325
Random forest 0.6317130730774614
Boosted trees 0.6334448040854458


## Error audit
These bands use outcomes only for post-test diagnostics, not prediction-time filtering. Feature associations do not establish causes.

In [7]:
pd.DataFrame(s['error_groups'])

,group,n,log_mae,mae_shares,median_actual,median_prediction
0,"Under 1,000",2917,0.754794,826.842520,784.0,1488.619151
1,"1,000–9,999",5951,0.447473,1139.617742,1700.0,1854.360539
2,"10,000+",444,2.292489,23783.888516,17150.0,2032.317692


In [8]:
pd.read_csv(ROOT / 'results/largest_errors.csv')

,row_id,date,shares,channel,Median baseline,Ridge,Random forest,Boosted trees,abs_log_error
0,38633,2014-12-10,5,socmed,1500.0,2274.736277,1609.202951,1874.103339,5.592356
1,35256,2014-10-22,284700,world,1500.0,1287.494798,1506.928916,1661.583104,5.240702
2,37590,2014-11-24,211600,Other / unlabelled,1500.0,2295.205638,1949.310618,1871.080159,4.686714
3,37873,2014-11-28,200100,Other / unlabelled,1500.0,2372.010182,1863.591016,1803.856061,4.675781
4,36995,2014-11-15,161600,Other / unlabelled,1500.0,2404.109755,1952.389426,1859.532907,4.415564
5,35327,2014-10-23,28,lifestyle,1500.0,1908.734599,2145.695871,2387.456027,4.304389
6,34724,2014-10-14,109500,entertainment,1500.0,1266.166741,1644.846102,1796.848666,4.197679
7,37542,2014-11-23,87600,entertainment,1500.0,1272.507391,1327.888268,1247.294414,4.188450


## Rerun the complete pipeline
Run from repository root in a terminal:
```bash
python src/analyze.py --data data/online+news+popularity.zip
python src/build_site.py
```
See README for installation, sources, limitations and the AI-use disclosure. For a future extension, collect fixed-horizon targets and perform rolling-origin validation; those experiments are not part of this run.